In [1]:
import re, csv
from pathlib import Path
from IPython.display import Markdown, display

FIG = Path("../results/figures")

# --- Parse summary statistics ---
summary_txt = (FIG / "ndvi_summary_statistics.txt").read_text()
unique_pas = re.search(r"Unique PAs:\s*([\d,]+)", summary_txt).group(1)
total_transects = re.search(r"Total transects:\s*([\d,]+)", summary_txt).group(1)
countries = re.search(r"Countries \(ISO3\):\s*(\d+)", summary_txt).group(1)
n_biomes = re.search(r"Biomes:\s*(\d+)", summary_txt).group(1)
biome_lines = re.findall(r"^\s+(.+?):\s*([\d,]+)", summary_txt, re.M)
top2 = " and ".join(f"{n} ({c} PAs)" for n, c in biome_lines[:2])
sig_inc = re.search(r"sig_increase:\s*([\d,]+)", summary_txt).group(1)
sig_dec = re.search(r"sig_decrease:\s*([\d,]+)", summary_txt).group(1)
neg_n = re.search(r"edge_intensity < 0:\s*([\d,]+)", summary_txt).group(1)
neg_pct = re.search(r"edge_intensity < 0:.*\(([\d.]+)%\)", summary_txt).group(1)

display(Markdown(
f"""## Summary Statistics

The analysis covers **{unique_pas} unique protected areas** across **{countries} countries** and **{n_biomes} biomes**, totalling **{total_transects} transects**. {top2} are the most represented biomes. Of all PAs, {sig_inc} show a statistically significant increase in edge extent over 2003–2025 and {sig_dec} show a significant decrease. {neg_pct}% of PAs ({neg_n}) exhibit negative mean edge intensity, indicating higher NDVI inside than outside the boundary."""
))

## Summary Statistics

The analysis covers **4,046 unique protected areas** across **146 countries** and **8 biomes**, totalling **1,736,434 transects**. Grassland & Shrubland (1,307 PAs) and Tropical Forest (1,166 PAs) are the most represented biomes. Of all PAs, 491 show a statistically significant increase in edge extent over 2003–2025 and 388 show a significant decrease. 66.8% of PAs (2,701) exhibit negative mean edge intensity, indicating higher NDVI inside than outside the boundary.

```{raw} latex
\clearpage
```

:::{figure} ../results/figures/ndvi_figure2a_extent_map_2025.png
:label: fig2a
:width: 100%
:::

:::{figure} ../results/figures/ndvi_figure2b_trend_map.png
:label: fig2b
:width: 100%
:::

```{raw} latex
{\small \noindent \textbf{Figure 2.} (a) Spatial distribution of edge extent across protected areas in 2025. (b) Significant temporal trends in edge extent (2003--2025); red = significant increase, blue = significant decrease, grey = not significant.}

\vspace{1em}
```

Edge extent varies widely across regions, with tropical and grassland PAs tending toward higher values. Significant increases cluster in sub-Saharan Africa and South America, while significant decreases are concentrated in boreal and temperate zones.

```{raw} latex
\clearpage
```

:::{figure} ../results/figures/ndvi_figure3_trends_by_biome.png
:label: fig3
:width: 100%
:::

```{raw} latex
{\small \noindent \textbf{Figure 3.} (a) Proportion of PAs per biome classified by edge-extent trend direction and significance. (b) Biome-specific rates of change in edge extent (slope per SD-year) from the mixed-effects trend model with 95\% confidence intervals.}

\vspace{1em}
```

Biome-specific temporal trends were estimated using a linear mixed-effects model:

$$\text{edge\_extent}_{ij} = \beta_0 + \beta_1 \, \text{year}_z + \boldsymbol{\beta}_2 \, \text{Biome}_i + \boldsymbol{\beta}_3 \, (\text{year}_z \times \text{Biome}_i) + u_j + \varepsilon_{ij}$$

where $u_j \sim \mathcal{N}(0, \sigma_u^2)$ is a random intercept for PA $j$. Grassland & Shrubland and Tropical Forest biomes show significant positive temporal trends in edge extent.

```{raw} latex
\clearpage
```

:::{figure} ../results/figures/ndvi_figure4_mixed_model_coefficients.png
:label: fig4
:width: 100%
:::

```{raw} latex
{\small \noindent \textbf{Figure 4.} (a) Standardised mixed-model coefficients for edge extent in 2025. (b) Standardised mixed-model coefficients for edge intensity in 2025. Error bars denote 95\% confidence intervals; the dashed red line marks zero.}

\vspace{1em}
```

Cross-sectional drivers of edge effects in 2025 were assessed using linear mixed-effects models of the form:

$$y_j = \beta_0 + \sum_{k} \beta_k \, x_{kj}^{(z)} + u_j + \varepsilon_j$$

where $y$ is edge extent or edge intensity, $x_k^{(z)}$ are z-scored covariates (status year, PA area, gHM, elevation, slope, water extent), and $u_j \sim \mathcal{N}(0, \sigma_u^2)$. Lower slope and elevation are related to higher edge extent and intensity, while higher water extent and human modification (gHM) outside of the park are associated with higher edge extent and intensity. PA area has a small but significant positive effect on edge intensity.

```{raw} latex
\clearpage
```

:::{figure} ../results/figures/ndvi_figure5a_correlation.png
:label: fig5a
:width: 100%
:::

:::{figure} ../results/figures/ndvi_figure5b_distributions.png
:label: fig5b
:width: 100%
:::

```{raw} latex
{\small \noindent \textbf{Figure 5.} (a) Scatter plot of edge extent versus edge intensity (Cohen's d) for all PAs in 2025. (b) Distributions of edge extent and edge intensity across all PAs.}

\vspace{1em}
```

Edge extent and intensity are positively correlated, with a saturation effect apparent in the highest values.

```{raw} latex
\clearpage
```

:::{figure} ../results/figures/ndvi_figure6_saturation.png
:label: fig6
:width: 100%
:::

```{raw} latex
{\small \noindent \textbf{Figure 6.} (a) Saturation analysis for edge extent --- relationship between 2003 baseline and annual slope. (b) Same for edge intensity. Red lines show OLS fits.}

\vspace{1em}
```

PAs that already had high edge extent or intensity in 2003 tend to have lower or negative slopes, consistent with a saturation effect where boundaries with initially strong edges have limited capacity for further increase.

In [2]:
# --- Parse ANOVA results ---
anova_txt = (FIG / "ndvi_anova_results.txt").read_text()

def parse_anova(txt, factor):
    block = txt.split(f"ANOVA for {factor}:")[1].split("ANOVA for")[0]
    for row in block.strip().split("\n"):
        cols = row.split()
        if len(cols) >= 5 and cols[0] not in ("sum_sq", "Residual", ""):
            try:
                df_num = int(float(cols[2]))
                f_val = float(cols[3])
                p_val = float(cols[4])
            except (ValueError, IndexError):
                continue
            for r2 in block.strip().split("\n"):
                if r2.strip().startswith("Residual"):
                    df_den = int(float(r2.split()[2]))
                    break
            return df_num, df_den, f_val, p_val
    return None

def fmt(label, factor):
    a = parse_anova(anova_txt, factor)
    if a:
        dn, dd, f, p = a
        ps = "< 0.001" if p < 0.001 else f"= {p:.3f}"
        return f"**{label}** ($F_{{{dn},\\,{dd}}} = {f:.1f}$, $p {ps}$)"
    return ""

anova_parts = ", followed by ".join([
    fmt("biome", "BIOME_NAME"), fmt("IUCN category", "IUCN_CAT"),
    fmt("PA area", "AREA_DISSO"), fmt("status year", "STATUS_YR"),
])

# --- CSV tables ---
def csv_to_md(filename, col_map):
    rows = list(csv.DictReader((FIG / filename).open()))
    keys, headers = zip(*col_map.items())
    lines = ["| " + " | ".join(headers) + " |",
             "|" + "|".join(["---"] * len(headers)) + "|"]
    for r in rows:
        vals = []
        for k in keys:
            v = r[k]
            if "p_value" in k:
                try: v = "<0.001" if float(v) < 0.001 else f"{float(v):.3f}"
                except ValueError: pass
            vals.append(v)
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)

t1 = csv_to_md("ndvi_table1_slopes.csv",
    {"WDPA_PID": "WDPA_PID", "BIOME_NAME": "Biome",
     "slope_extent": "Slope", "p_value_extent": "p-value"})
t2 = csv_to_md("ndvi_table2_edge_extent_2025.csv",
    {"WDPA_PID": "WDPA_PID", "BIOME_NAME": "Biome", "edge_extent": "Edge Extent"})
t3 = csv_to_md("ndvi_table3_edge_intensity_2025.csv",
    {"WDPA_PID": "WDPA_PID", "BIOME_NAME": "Biome", "edge_intensity": "Edge Intensity"})

display(Markdown(
f"""## ANOVA Results

One-way ANOVAs on edge extent confirm that {anova_parts}. All four factors are highly significant predictors of edge extent.

## Tables

:::{{table}} Top 25 PAs by edge-extent slope (significant increase only).
:label: tab1

{t1}
:::

:::{{table}} Top 25 PAs by edge extent in 2025.
:label: tab2

{t2}
:::

:::{{table}} Top 25 PAs by edge intensity (Cohen's d) in 2025.
:label: tab3

{t3}
:::"""
))

## ANOVA Results

One-way ANOVAs on edge extent confirm that **biome** ($F_{7,\,92867} = 341.8$, $p < 0.001$), followed by **IUCN category** ($F_{9,\,92865} = 97.9$, $p < 0.001$), followed by **PA area** ($F_{1,\,92873} = 176.9$, $p < 0.001$), followed by **status year** ($F_{1,\,92873} = 77.9$, $p < 0.001$). All four factors are highly significant predictors of edge extent.

## Tables

:::{table} Top 25 PAs by edge-extent slope (significant increase only).
:label: tab1

| WDPA_PID | Biome | Slope | p-value |
|---|---|---|---|
| 28556 | Grassland & Shrubland | 0.0123 | <0.001 |
| 34032 | Tropical Forest | 0.0117 | <0.001 |
| 34033 | Tropical Forest | 0.0116 | <0.001 |
| 313442 | Grassland & Shrubland | 0.0111 | <0.001 |
| 555711941 | Grassland & Shrubland | 0.0105 | <0.001 |
| 352274 | Tropical Forest | 0.0097 | <0.001 |
| 555558302 | Grassland & Shrubland | 0.0095 | <0.001 |
| 1091 | Grassland & Shrubland | 0.0095 | <0.001 |
| 31758 | Grassland & Shrubland | 0.0092 | <0.001 |
| 555710747 | Boreal Forest | 0.0088 | <0.001 |
| 3462 | Tropical Forest | 0.0087 | <0.001 |
| 34013 | Tropical Forest | 0.0084 | <0.001 |
| 313019 | Tropical Forest | 0.0083 | <0.001 |
| 555542970 | Grassland & Shrubland | 0.0081 | <0.001 |
| 13768 | Grassland & Shrubland | 0.0079 | <0.001 |
| 555623807 | Grassland & Shrubland | 0.0072 | <0.001 |
| 68861 | Tropical Forest | 0.007 | <0.001 |
| 351898 | Grassland & Shrubland | 0.0067 | 0.007 |
| 902847 | Temperate Forest | 0.0067 | 0.001 |
| 354013 | Tropical Forest | 0.0066 | <0.001 |
| 352709 | Grassland & Shrubland | 0.0066 | <0.001 |
| 1293 | Grassland & Shrubland | 0.0065 | <0.001 |
| 33617 | Tropical Forest | 0.0064 | <0.001 |
| 29615 | Grassland & Shrubland | 0.0064 | <0.001 |
| 1440 | Grassland & Shrubland | 0.0063 | <0.001 |
:::

:::{table} Top 25 PAs by edge extent in 2025.
:label: tab2

| WDPA_PID | Biome | Edge Extent |
|---|---|---|
| 81051 | Tropical Forest | 0.9685 |
| 18689 | Grassland & Shrubland | 0.9576 |
| 28576 | Grassland & Shrubland | 0.9469 |
| 354872 | Temperate Forest | 0.942 |
| 60 | Tropical Forest | 0.927 |
| 64360 | Desert | 0.9144 |
| 8851 | Tropical Forest | 0.9133 |
| 64364 | Desert | 0.9057 |
| 17923 | Tropical Forest | 0.897 |
| 34016 | Tropical Forest | 0.8939 |
| 902690 | Grassland & Shrubland | 0.8819 |
| 555576419 | Grassland & Shrubland | 0.8733 |
| 354013 | Tropical Forest | 0.8559 |
| 19667 | Tropical Forest | 0.8538 |
| 35 | Tropical Forest | 0.8536 |
| 19769 | Tropical Forest | 0.8483 |
| 7730 | Tropical Forest | 0.8431 |
| 315143 | Tropical Forest | 0.8387 |
| 7693 | Tropical Forest | 0.8319 |
| 33184 | Grassland & Shrubland | 0.8217 |
| 555719282 | Temperate Forest | 0.8203 |
| 301425 | Grassland & Shrubland | 0.8194 |
| 34037 | Tropical Forest | 0.8191 |
| 351815 | Tropical Forest | 0.8172 |
| 64112 | Desert | 0.807 |
:::

:::{table} Top 25 PAs by edge intensity (Cohen's d) in 2025.
:label: tab3

| WDPA_PID | Biome | Edge Intensity |
|---|---|---|
| 354872 | Temperate Forest | 2.4293 |
| 18689 | Grassland & Shrubland | 1.8794 |
| 28576 | Grassland & Shrubland | 1.8146 |
| 8851 | Tropical Forest | 1.6523 |
| 64364 | Desert | 1.4758 |
| 64360 | Desert | 1.3508 |
| 67730 | Temperate Forest | 1.2955 |
| 902690 | Grassland & Shrubland | 1.2802 |
| 7730 | Tropical Forest | 1.2666 |
| 555576419 | Grassland & Shrubland | 1.1591 |
| 81051 | Tropical Forest | 0.9997 |
| 17923 | Tropical Forest | 0.9879 |
| 12186 | Mangrove | 0.9785 |
| 555542759 | Tropical Forest | 0.9701 |
| 34016 | Tropical Forest | 0.9573 |
| 35 | Tropical Forest | 0.913 |
| 60 | Tropical Forest | 0.8881 |
| 555731237 | Temperate Forest | 0.8682 |
| 33984 | Grassland & Shrubland | 0.8433 |
| 555558302 | Grassland & Shrubland | 0.8278 |
| 351776 | Tropical Forest | 0.8188 |
| 7693 | Tropical Forest | 0.803 |
| 18893 | Tropical Forest | 0.7954 |
| 352261 | Tropical Forest | 0.7935 |
| 352420 | Grassland & Shrubland | 0.7914 |
:::

```{raw} latex
\clearpage
```

## Supplementary Figures

:::{figure} ../results/figures/ndvi_edge_extent_model_diagnostics.png
:label: figs1
:width: 100%
:::

```{raw} latex
{\small \noindent \textbf{Figure S1.} Diagnostics for the edge-extent mixed model: observed vs.\ fitted values (left) and residual distribution (right).}

\vspace{1em}
```

Residuals are approximately normally distributed and the observed-vs-fitted plot shows reasonable agreement, supporting model adequacy for edge extent.

```{raw} latex
\clearpage
```

:::{figure} ../results/figures/ndvi_edge_intensity_model_diagnostics.png
:label: figs2
:width: 100%
:::

```{raw} latex
{\small \noindent \textbf{Figure S2.} Diagnostics for the edge-intensity mixed model: observed vs.\ fitted values (left) and residual distribution (right).}

\vspace{1em}
```

The edge-intensity model shows greater residual spread, reflecting higher variability in Cohen's d across PAs, but residuals remain approximately symmetric.